In [ ]:
# ═══════════════════════════════════════════════════════════
# GPU VERIFICATION
# Check if TensorFlow is successfully using a GPU accelerator.
# ═══════════════════════════════════════════════════════════
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) == 0:
    print("⚠️ WARNING: No GPU detected. CNN will run on CPU.")
    print("   Check Kaggle notebook Settings > Accelerator")
else:
    print(f"✅ GPU detected: {gpus}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


## 🔧 Environment Setup

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT SETUP
# Install libraries, pull latest pipeline code from GitHub,
# create output folder structure, define session config.
# Run this once at the start of every weekly session.
# ═══════════════════════════════════════════════════════════
import os, sys, subprocess, shutil
from datetime import date

# Detect if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')

# ── User config ─────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/Iceandlava124/tess-exoplanet-pipeline.git"
KAGGLE_DATASET = "bhavishmehta/tess-exoplanet-discovery-results"
SESSION_LABEL  = date.today().strftime("%Y-%m-%d")
TIME_LIMIT_HRS = 5.0
DISK_LIMIT_GB  = 18
STARS_PER_RUN  = 400

if IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"
    PIPELINE_DIR = os.path.join(WORKING_DIR, "pipeline")
    RESULTS_DIR  = os.path.join(WORKING_DIR, "results")
else:
    WORKING_DIR = os.path.abspath(".")
    PIPELINE_DIR = WORKING_DIR
    RESULTS_DIR  = os.path.join(WORKING_DIR, "results")

# ── Install libraries quietly ────────────────────────────────
if IS_KAGGLE:
    print("Installing libraries...")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "lightkurve", "wotan", "astropy", "astroquery",
             "batman-package", "transitleastsquares", "ldtk", "scipy", "scikit-learn",
             "imbalanced-learn", "tqdm", "joblib",
             "pandas", "numpy"],
            check=True, capture_output=True
        )
        print("Libraries installed.")
    except Exception as e:
        print(f"Library install warning (may already be installed): {e}")
else:
    print("Running locally. Skipping pip install cell.")

# ── Pull latest pipeline code from GitHub ───────────────────
if IS_KAGGLE:
    try:
        if os.path.isdir(PIPELINE_DIR):
            print("Updating existing pipeline clone...")
            result = subprocess.run(
                ["git", "-C", PIPELINE_DIR, "pull"],
                capture_output=True, text=True, timeout=120
            )
            print(result.stdout.strip() or "Already up to date.")
        else:
            print("Cloning pipeline repository...")
            result = subprocess.run(
                ["git", "clone", "--depth=1", GITHUB_REPO, PIPELINE_DIR],
                capture_output=True, text=True, timeout=180
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr.strip())
            print("Pipeline cloned successfully.")
    except Exception as e:
        print(f"❌ Git operation failed: {e}")
        if not os.path.isdir(PIPELINE_DIR) or len(os.listdir(PIPELINE_DIR)) == 0:
            raise RuntimeError(
                "Pipeline directory is missing or empty after git "
                "failure — cannot continue. Fix the GitHub clone "
                "before running further cells."
            )
        print("   Continuing with existing pipeline/ directory.")
else:
    print("Running locally. Using current directory codebase.")

# ── Add pipeline to Python path ──────────────────────────────
for p in [PIPELINE_DIR, WORKING_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Create output folder structure ──────────────────────────
for folder in [
    os.path.join(RESULTS_DIR, "KEEP"),
    os.path.join(RESULTS_DIR, "FLAG"),
    os.path.join(RESULTS_DIR, "DISCARD"),
    os.path.join(RESULTS_DIR, "plots", "flag_analysis"),
    os.path.join(RESULTS_DIR, "reports"),
    os.path.join(RESULTS_DIR, "figures"),
]:
    os.makedirs(folder, exist_ok=True)

# ── Resource Copying (Models & Caches) ──────────────────────────
def find_kaggle_input_dir(slug: str) -> str:
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if slug in os.path.basename(root):
                return root
    return os.path.join('/kaggle/input', slug)

if IS_KAGGLE:
    INPUT_DIR = find_kaggle_input_dir("exoplanet-pipeline-resources")
    if os.path.exists(INPUT_DIR):
        print(f"Loading models, target catalogs, and cached queries from: {INPUT_DIR}")
        
        # Ensure target directories exist
        os.makedirs(os.path.join(WORKING_DIR, "models"), exist_ok=True)
        os.makedirs(os.path.join(PIPELINE_DIR, "models"), exist_ok=True)
        os.makedirs(os.path.join(WORKING_DIR, "data"), exist_ok=True)
        os.makedirs(os.path.join(PIPELINE_DIR, "data"), exist_ok=True)
        
        # Copy models
        for m in ["random_forest.pkl", "cnn_classifier.h5"]:
            src_m = os.path.join(INPUT_DIR, m)
            if os.path.exists(src_m):
                shutil.copy2(src_m, os.path.join(WORKING_DIR, "models", m))
                shutil.copy2(src_m, os.path.join(PIPELINE_DIR, "models", m))
                print(f"Loaded model: {m}")
        
        # Copy caching/data files
        for d in ["training_targets.csv", "pipeline_cache.db"]:
            src_d = os.path.join(INPUT_DIR, d)
            if os.path.exists(src_d):
                shutil.copy2(src_d, os.path.join(WORKING_DIR, "data", d))
                shutil.copy2(src_d, os.path.join(PIPELINE_DIR, "data", d))
                print(f"Loaded data: {d}")
        print("SUCCESS: Resources loaded.")
    else:
        print(f"Warning: Kaggle resources directory not found at {INPUT_DIR}")

# ── Configure Kaggle API Credentials from Secrets ────────────────
try:
    from kaggle_secrets import UserSecretsClient
    import json
    user_secrets = UserSecretsClient()
    k_user = user_secrets.get_secret("KAGGLE_USERNAME")
    k_key = user_secrets.get_secret("KAGGLE_KEY")
    if k_user and k_key:
        dot_kaggle = os.path.expanduser("~/.kaggle")
        os.makedirs(dot_kaggle, exist_ok=True)
        with open(os.path.join(dot_kaggle, "kaggle.json"), "w") as f:
            json.dump({"username": k_user, "key": k_key}, f)
        os.chmod(os.path.join(dot_kaggle, "kaggle.json"), 0o600)
        print("✅ Kaggle API credentials successfully configured from secrets.")
    else:
        print("⚠️ Kaggle secrets not found. Dataset pushes may fail.")
except Exception as e:
    print(f"⚠️ Kaggle secrets configuration skipped: {e}")

# ── Copy dataset metadata so that Kaggle CLI pushes work ────────────────
src_meta = os.path.join(PIPELINE_DIR, "results", "dataset-metadata.json")
dest_meta = os.path.join(RESULTS_DIR, "dataset-metadata.json")
if os.path.exists(src_meta):
    shutil.copy2(src_meta, dest_meta)
    print("✅ Dataset metadata copied to results directory.")
else:
    # Fallback: generate it if missing
    import json
    meta = {
        "title": "TESS Exoplanet Discovery Results",
        "id": KAGGLE_DATASET,
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(dest_meta, "w") as f:
        json.dump(meta, f, indent=4)
    print("✅ Generated dataset-metadata.json in results directory.")


## 📂 Resume From Last Week

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — RESUME FROM LAST WEEK
# If previous results exist, skip already processed stars.
# ═══════════════════════════════════════════════════════════
import pandas as pd
import csv
import os
import shutil
from collections import Counter

# Diagnostic check at the top
print("Contents of /kaggle/input/:")
if os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        print(f"   - {item}")
else:
    print("   /kaggle/input does not exist")

if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

# Reuse Cell 1's input dir search pattern
def find_results_input_dir():
    if not os.path.exists('/kaggle/input'):
        return None
    for item in os.listdir('/kaggle/input'):
        if 'tess-exoplanet-discovery-results' in item:
            return os.path.join('/kaggle/input', item)
    return None

if IS_KAGGLE:
    found_dir = find_results_input_dir()
    if found_dir:
        INPUT_RESULTS_CSV = os.path.join(found_dir, "results.csv")
        INPUT_CANDIDATES = os.path.join(found_dir, "candidates_submission.csv")
        INPUT_DISCOVERY_LOG = os.path.join(found_dir, "DISCOVERY_LOG.md")
        print(f"📂 Found results input at: {found_dir}")
    else:
        INPUT_RESULTS_CSV = None
        INPUT_CANDIDATES = None
        INPUT_DISCOVERY_LOG = None
        print("⚠️ No previous results dataset found attached to this notebook.")
        print("   Check: is 'tess-exoplanet-discovery-results' added as an")
        print("   Input to this notebook via Kaggle's 'Add Input' panel?")
else:
    INPUT_RESULTS_CSV = os.path.join(RESULTS_DIR, "results.csv")
    INPUT_CANDIDATES = os.path.join(RESULTS_DIR, "candidates_submission.csv")
    INPUT_DISCOVERY_LOG = os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md")

OUTPUT_RESULTS_CSV = os.path.join(RESULTS_DIR, "results.csv")
already_done = set()
df_previous  = None

COLUMNS_22 = [
    "tic_id", "session_label", "decision", "final_class", "confidence",
    "period", "period_err", "depth", "depth_err", "duration",
    "duration_err", "snr", "flag_reasons", "rp_earth", "is_new_discovery",
    "alias_rejected", "fpp", "combined_fpp", "fpp_status",
    "contamination_ratio", "n_nearby_gaia_stars", "n_sectors_consistent"
]

try:
    if INPUT_RESULTS_CSV is not None and os.path.exists(INPUT_RESULTS_CSV):
        print(f"📂 Found previous results at: {INPUT_RESULTS_CSV}")
        # Parse using python csv module to be robust against schema/column count variations
        rows = []
        with open(INPUT_RESULTS_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            try:
                header = next(reader)
            except StopIteration:
                header = []
            
            for row in reader:
                if not row:
                    continue
                
                # Map row to 22 columns
                new_row = {}
                if len(row) == 12:
                    new_row["tic_id"] = row[0]
                    new_row["session_label"] = ""
                    new_row["decision"] = row[1]
                    new_row["final_class"] = row[2]
                    new_row["confidence"] = row[3]
                    new_row["period"] = row[4]
                    new_row["period_err"] = row[5]
                    new_row["depth"] = row[6]
                    new_row["depth_err"] = row[7]
                    new_row["duration"] = row[8]
                    new_row["duration_err"] = row[9]
                    new_row["snr"] = row[10]
                    new_row["flag_reasons"] = row[11]
                    # Defaults for new columns
                    new_row["rp_earth"] = "0.0"
                    new_row["is_new_discovery"] = "False"
                    new_row["alias_rejected"] = "False"
                    new_row["fpp"] = "None"
                    new_row["combined_fpp"] = "None"
                    new_row["fpp_status"] = "skipped"
                    new_row["contamination_ratio"] = "None"
                    new_row["n_nearby_gaia_stars"] = "0"
                    new_row["n_sectors_consistent"] = "1"
                else:
                    # Pad or truncate to 22 fields
                    for i, col in enumerate(COLUMNS_22):
                        new_row[col] = row[i] if i < len(row) else ""
                rows.append(new_row)
        
        # Deduplicate by tic_id keeping the LAST entry
        unique_rows = {}
        for r in rows:
            if r["tic_id"]:
                try:
                    unique_rows[int(r["tic_id"])] = r
                except ValueError:
                    pass
                    
        dedup_rows = list(unique_rows.values())
        
        # Write standardized file to OUTPUT_RESULTS_CSV
        with open(OUTPUT_RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=COLUMNS_22)
            writer.writeheader()
            writer.writerows(dedup_rows)
            
        # Load into pandas
        df_previous = pd.read_csv(OUTPUT_RESULTS_CSV)
        if "tic_id" in df_previous.columns:
            already_done = set(df_previous["tic_id"].astype(int).tolist())
            
        print(f"📋 Resuming — Loaded {len(df_previous)} previous results.")
        print(f"   Already processed: {len(already_done)} unique stars")
        if "decision" in df_previous.columns:
            counts = Counter(df_previous["decision"].fillna("UNKNOWN"))
            print(f"   KEEP:    {counts.get('KEEP', 0)}")
            print(f"   FLAG:    {counts.get('FLAG', 0)}")
            print(f"   DISCARD: {counts.get('DISCARD', 0)}")
    else:
        if INPUT_RESULTS_CSV is None:
            print("🆕 Fresh start — no previous results dataset is attached to this notebook")
        else:
            print("🆕 Fresh start — results dataset is attached but results.csv was not found inside it")
        already_done = set()
except Exception as e:
    print(f"❌ Error loading previous results: {e}")
    already_done = set()
    print("   Continuing with empty already_done set.")

# Restore DISCOVERY_LOG.md
try:
    if INPUT_DISCOVERY_LOG and os.path.exists(INPUT_DISCOVERY_LOG) and IS_KAGGLE:
        shutil.copy2(
            INPUT_DISCOVERY_LOG,
            os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md")
        )
        print("   Discovery log history restored.")
    else:
        print("   No previous discovery log found — will start fresh.")
except Exception as e:
    print(f"   Warning: could not restore discovery log: {e}")

try:
    if INPUT_CANDIDATES is not None and os.path.exists(INPUT_CANDIDATES):
        shutil.copy2(INPUT_CANDIDATES, os.path.join(RESULTS_DIR, "candidates_submission.csv"))
        print("   Candidates file copied.")
except Exception as e:
    print(f"   Warning: could not copy candidates file: {e}")

print(f"✅ Resume check complete — {len(already_done)} stars will be skipped.")


## 🎯 This Week's Targets

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — BUILD TARGET LIST
# ═══════════════════════════════════════════════════════════
import sys, os
if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

for path_dir in ["/kaggle/working/pipeline", "/kaggle/working", os.path.abspath(".")]:
    if path_dir not in sys.path:
        sys.path.insert(0, path_dir)

try:
    from kaggle_discovery_runner import build_target_list
except Exception as e_imp:
    print(f"❌ Failed to import build_target_list: {e_imp}")
    print("Python Path is:", sys.path)
    raise e_imp

TARGETS_CSV = os.path.join(RESULTS_DIR, "this_week_targets.csv")

try:
    targets = build_target_list(
        n_targets            = STARS_PER_RUN,
        already_processed    = already_done if 'already_done' in globals() else set(),
        mag_range            = (8, 13),
        teff_range           = (3500, 7000),
        radius_range         = (0.5, 2.0),
        exclude_giants       = True,
        exclude_known_contaminated = True,
        prioritise_multi_sector    = True,
        prioritise_not_in_toi      = True,
    )
    targets.to_csv(TARGETS_CSV, index=False)
    n = len(targets)
    est_hrs = n * 45 / 3600
    print(f"🎯 {n} stars queued for this session")
    print(f"   Estimated time: {est_hrs:.1f} hours at ~45s per star")
    print(f"   Target list saved to: {TARGETS_CSV}")
except Exception as e:
    print(f"❌ Target list build failed: {e}")
    import traceback; traceback.print_exc()
    if not already_done:
        raise RuntimeError(
            "Target list build failed and already_done was empty — cannot continue. "
            "Check network connection and astroquery/MAST service status."
        )
    targets = pd.DataFrame(columns=["tic_id"])

print(f"✅ Target list ready — {len(targets)} stars")


## 🔭 Autonomous Discovery — ~8.5 hrs

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — AUTONOMOUS DISCOVERY
# This is the main cell. It runs the full 13-step pipeline
# on every star in the target list. It stops automatically
# when time or disk limits are hit. One bad star never
# crashes the entire run.
# Expected runtime: 6-9 hours on Kaggle GPU.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import run_discovery_session
except ImportError:
    sys.path.insert(0, WORKING_DIR)
    from kaggle_discovery_runner import run_discovery_session

session_summary = {}
if len(targets) == 0:
    print("❌ No targets to process — re-run Cell 3 first.")
else:
    try:
        session_summary = run_discovery_session(
            targets           = targets,
            output_dir        = RESULTS_DIR,
            models_dir        = os.path.join(WORKING_DIR, "models"),
            time_limit_hours  = TIME_LIMIT_HRS,
            disk_limit_gb     = DISK_LIMIT_GB,
            save_every_n      = 50,
            session_label     = SESSION_LABEL,
            # Pipeline feature flags
            run_flag_analyzer      = True,
            run_candidate_export   = True,
            run_toi_crosscheck     = True,
            alias_rejection        = True,
            # Physical plausibility filters
            max_planet_radius_earth = 25.0,
            min_transit_snr         = 5.0,
            min_depth_ppm           = 100,
            # Log file
            log_file = os.path.join(RESULTS_DIR, "discovery_log.txt"),
        )
        n_proc    = session_summary.get("stars_processed", 0)
        n_skip    = session_summary.get("stars_skipped", 0)
        elapsed   = session_summary.get("elapsed_hours", 0)
        print(f"✅ Discovery session complete")
        print(f"   Stars processed: {n_proc}")
        print(f"   Stars skipped:   {n_skip}")
        print(f"   Time elapsed:    {elapsed:.1f} hrs")
    except Exception as e:
        print(f"❌ Discovery session crashed: {e}")
        import traceback; traceback.print_exc()
        session_summary = {}

## 📊 Session Summary

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — SESSION SUMMARY
# Print the final statistics table, highlight any potential
# new planet discoveries not in the TOI catalog, and append
# one line to the cumulative DISCOVERY_LOG.md.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import generate_session_summary
except ImportError:
    sys.path.insert(0, WORKING_DIR)
    from kaggle_discovery_runner import generate_session_summary

try:
    summary = generate_session_summary(
        results_dir   = RESULTS_DIR,
        session_label = SESSION_LABEL,
        session_data  = session_summary,
    )

    n_attempted  = summary.get("stars_attempted", 0)
    n_processed  = summary.get("stars_processed", 0)
    n_skipped    = summary.get("stars_skipped", 0)
    n_alias      = summary.get("alias_discards", 0)
    elapsed      = summary.get("elapsed_hours", 0)
    throughput   = n_processed / elapsed if elapsed > 0 else 0
    n_keep       = summary.get("keep", 0)
    n_flag       = summary.get("flag", 0)
    n_discard    = summary.get("discard", 0)
    n_new        = summary.get("new_discoveries", 0)
    n_cumulative = summary.get("cumulative_total", 0)

    print("=" * 50)
    print(f"  TESS DISCOVERY RUN -- {SESSION_LABEL}")
    print("=" * 50)
    print(f"  Stars attempted:   {n_attempted}")
    print(f"  Stars processed:   {n_processed}")
    print(f"  Skipped (no data): {n_skipped}")
    print(f"  Alias discards:    {n_alias}")
    print(f"  Time elapsed:      {elapsed:.1f} hrs")
    print(f"  Throughput:        {throughput:.0f} stars/hr")
    print("-" * 50)
    print(f"  KEEP:    {n_keep} ({n_new} potential new discoveries)")
    print(f"  FLAG:    {n_flag} (human review needed)")
    print(f"  DISCARD: {n_discard}")
    print("-" * 50)
    print(f"  Cumulative total:  {n_cumulative} stars all-time")
    print("=" * 50)
except Exception as e:
    print(f"❌ Summary generation failed: {e}")
    summary = {}
    n_new = 0
    n_cumulative = 0

# Print any potential new discoveries not in the TOI catalog.
try:
    new_disc_file = os.path.join(RESULTS_DIR, "new_discoveries.txt")
    if os.path.exists(new_disc_file) and os.path.getsize(new_disc_file) > 0:
        with open(new_disc_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        if lines:
            print("")
            print("POTENTIAL NEW DISCOVERIES NOT IN TOI CATALOG:")
            for line in lines:
                print(f"  {line}")
except Exception as e:
    print(f"   Warning: could not read new_discoveries.txt: {e}")

# Append one row to the cumulative discovery log markdown table.
try:
    log_path = os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md")
    write_header = not os.path.exists(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        if write_header:
            f.write("# TESS Discovery Log\n\n")
            f.write("| Date | Processed | KEEP | FLAG | DISCARD | New | Notes |\n")
            f.write("|------|-----------|------|------|---------|-----|-------|\n")
        notes = f"Alias discards: {summary.get('alias_discards', 0)}"
        f.write(f"| {SESSION_LABEL} | {n_processed} | {summary.get('keep',0)} | "
                f"{summary.get('flag',0)} | {summary.get('discard',0)} | "
                f"{summary.get('new_discoveries',0)} | {notes} |\n")
    print("✅ Discovery log updated")
except Exception as e:
    print(f"❌ Log update failed: {e}")

## 💾 Save & Export

In [ ]:

# ── Deduplicate results.csv before exporting ────────────────
results_csv_file = os.path.join(RESULTS_DIR, "results.csv")
if os.path.exists(results_csv_file):
    try:
        import csv
        COLUMNS_22 = [
            "tic_id", "session_label", "decision", "final_class", "confidence",
            "period", "period_err", "depth", "depth_err", "duration",
            "duration_err", "snr", "flag_reasons", "rp_earth", "is_new_discovery",
            "alias_rejected", "fpp", "combined_fpp", "fpp_status",
            "contamination_ratio", "n_nearby_gaia_stars", "n_sectors_consistent"
        ]
        rows = []
        with open(results_csv_file, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = next(reader, [])
            for r in reader:
                if not r:
                    continue
                new_r = {}
                if len(r) == 12:
                    new_r["tic_id"] = r[0]
                    new_r["session_label"] = ""
                    new_r["decision"] = r[1]
                    new_r["final_class"] = r[2]
                    new_r["confidence"] = r[3]
                    new_r["period"] = r[4]
                    new_r["period_err"] = r[5]
                    new_r["depth"] = r[6]
                    new_r["depth_err"] = r[7]
                    new_r["duration"] = r[8]
                    new_r["duration_err"] = r[9]
                    new_r["snr"] = r[10]
                    new_r["flag_reasons"] = r[11]
                    new_r["rp_earth"] = "0.0"
                    new_r["is_new_discovery"] = "False"
                    new_r["alias_rejected"] = "False"
                    new_r["fpp"] = "None"
                    new_r["combined_fpp"] = "None"
                    new_r["fpp_status"] = "skipped"
                    new_r["contamination_ratio"] = "None"
                    new_r["n_nearby_gaia_stars"] = "0"
                    new_r["n_sectors_consistent"] = "1"
                else:
                    for i, col in enumerate(COLUMNS_22):
                        new_r[col] = r[i] if i < len(r) else ""
                rows.append(new_r)
        
        unique_rows = {}
        duplicate_count = 0
        for r in rows:
            if r["tic_id"]:
                tic_int = int(r["tic_id"])
                if tic_int in unique_rows:
                    duplicate_count += 1
                unique_rows[tic_int] = r
        
        dedup_rows = list(unique_rows.values())
        if duplicate_count > 0:
            print(f"⚠️ WARNING: {duplicate_count} duplicate TIC IDs found in results.csv — likely re-processed across sessions")
            with open(results_csv_file, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=COLUMNS_22)
                writer.writeheader()
                writer.writerows(dedup_rows)
            print(f"   Deduplicated: {len(dedup_rows)} unique stars remain")
        else:
            print(f"✅ Deduplication check complete: 0 duplicates found. {len(dedup_rows)} unique stars.")
    except Exception as e_dedup:
        print(f"⚠️ Warning during deduplication check: {e_dedup}")
# ═══════════════════════════════════════════════════════════
# CELL 6 — SAVE & EXPORT
# ═══════════════════════════════════════════════════════════
import subprocess
if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

export_map = {
    os.path.join(RESULTS_DIR, "results.csv"):                   "results_cumulative.csv",
    os.path.join(RESULTS_DIR, "candidates_submission.csv"):     "candidates_all.csv",
    os.path.join(RESULTS_DIR, "manual_review_queue.csv"):       "review_queue_latest.csv",
    os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md"):              "DISCOVERY_LOG.md",
    os.path.join(RESULTS_DIR, "new_discoveries.txt"):           "new_discoveries.txt",
}

for src, dest_name in export_map.items():
    try:
        if os.path.exists(src) and os.path.getsize(src) > 0:
            dest = os.path.join(WORKING_DIR, dest_name)
            shutil.copy2(src, dest)
            print(f"Saved: {dest_name}")
        else:
            print(f"   Skipped (empty or missing): {dest_name}")
    except Exception as e:
        print(f"   Failed to copy {dest_name}: {e}")

if IS_KAGGLE:
    try:
        n_processed_final = session_summary.get("stars_processed", 0)
        commit_msg = f"Auto-update: {SESSION_LABEL} -- {n_processed_final} stars processed"
        print(f"Pushing results to Kaggle dataset '{KAGGLE_DATASET}'...")
        result = subprocess.run(
            ["kaggle", "datasets", "version",
             "-p", RESULTS_DIR,
             "-m", commit_msg,
             "--dir-mode", "zip"],
            capture_output=True, text=True, timeout=300
        )
        print(f"   [kaggle CLI stdout]: {result.stdout.strip()}")
        print(f"   [kaggle CLI stderr]: {result.stderr.strip()}")
        print(f"   [kaggle CLI returncode]: {result.returncode}")

        if result.returncode == 0:
            print("✅ Dataset updated successfully.")
            push_succeeded = True
        else:
            print("❌ KAGGLE PUSH FAILED — results saved locally only.")
            print("   This means NEXT session will NOT see this run's data")
            print("   unless this is fixed before the next run.")
            push_succeeded = False
    except Exception as e:
        print(f"❌ Kaggle push raised an exception: {e}")
        push_succeeded = False
else:
    print("Running locally. Skipping Kaggle dataset upload.")
    push_succeeded = None

try:
    n_new_final  = summary.get("new_discoveries", 0)
    n_cumulative = summary.get("cumulative_total", 0)
except Exception:
    n_new_final  = 0
    n_cumulative = 0

if push_succeeded is False:
    print("\n⚠️⚠️⚠️ WARNING: THIS SESSION'S RESULTS WERE NOT SAVED")
    print("   TO THE PERSISTENT DATASET. Check the error above")
    print("   before running the next session, or this session's")
    print("   work will be invisible to future resume checks.")
else:
    print(f"\nAll done for {SESSION_LABEL}!")
    print(f"   New discoveries: {n_new_final}")
    print(f"   Cumulative stars analysed: {n_cumulative}")
    print("✅ Export complete")
